# Experiment: Recipient Evidence Card Gate

Objective: classify future recipient candidates from public evidence only, without names, contacts, private replies, patient data, samples, treatment, trial, purchase, import, or procurement scope.


In [ ]:
from __future__ import annotations

READY = "recipient_evidence_card_preclinical_quote_candidate"
HOLD = "recipient_evidence_card_hold_missing_capability"
WRONG_OWNER = "recipient_evidence_card_wrong_owner_hold"
BLOCKED = "recipient_evidence_card_release_blocked"

REQUIRED = {
    "public_source_urls",
    "recipient_class_allowed",
    "model_capability",
    "hbf_or_hbg_capability",
    "maturation_viability_capability",
    "hemolysis_or_heme_safety",
    "raw_data_export",
    "cost_timeline_constraints",
    "ethics_biosafety_constraints",
}
BLOCKERS = {
    "real_identity_in_public",
    "private_reply_in_public",
    "patient_records",
    "patient_samples",
    "treatment_or_dosing",
    "trial_screening",
    "purchase_import_procurement",
    "source_free_claim",
}


def classify_recipient_card(card: dict[str, bool | str]) -> str:
    """Return the public-safe label for a recipient evidence card."""
    if any(card.get(flag, False) for flag in BLOCKERS):
        return BLOCKED
    if card.get("recipient_class") == "clinical_trial_or_therapy_center":
        return WRONG_OWNER
    if any(not card.get(flag, False) for flag in REQUIRED):
        return HOLD
    return READY


base_card = {flag: True for flag in REQUIRED}
base_card["recipient_class"] = "erythroid_hbf_screening_lab"

cases = {
    "ready": base_card,
    "missing_model": {**base_card, "model_capability": False},
    "wrong_owner": {**base_card, "recipient_class": "clinical_trial_or_therapy_center"},
    "private_identity": {**base_card, "real_identity_in_public": True},
    "source_free": {**base_card, "source_free_claim": True},
}
expected = {
    "ready": READY,
    "missing_model": HOLD,
    "wrong_owner": WRONG_OWNER,
    "private_identity": BLOCKED,
    "source_free": BLOCKED,
}
results = {name: classify_recipient_card(card) for name, card in cases.items()}
assert results == expected
current_state = "template_ready_no_recipient_selected"

boundary = (
    "recipient card classifies capability only; it does not select or contact a lab"
)

decision_summary = {
    "current_state": current_state,
    "ready_label": READY,
    "boundary": boundary,
}
decision_summary

## Decision

Use the recipient evidence card template for future candidates. Keep real identities and contact details outside the public repo.
